# Tahap 4: Case Solution Reuse
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Gunakan putusan lama sebagai dasar prediksi solusi untuk kasus baru

### Alur Notebook:
1. Install & Import
2. Load Artefak dari Tahap 3
3. Fungsi Ekstraksi Solusi dari Top-K Kasus
4. Algoritma Prediksi (Majority Vote + Weighted Similarity)
5. Fungsi `predict_outcome()` Terpadu
6. Demo Manual 5 Kasus Baru
7. Simpan Hasil Prediksi ke `predictions.csv`

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q scikit-learn sentence-transformers pandas numpy joblib groq

## Cell 2 — Import Library

In [ ]:
import json
import re
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from getpass import getpass

import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from groq import Groq

warnings.filterwarnings("ignore")
print("✅ Library berhasil diimport")

## Cell 3 — Konfigurasi

In [ ]:
CONFIG = {
    "TOP_K"          : 5,
    "RETRIEVAL_MODE" : "tfidf",   # "tfidf" atau "embedding"
    "GROQ_MODEL"     : "llama-3.3-70b-versatile",
    "GROQ_DELAY"     : 1.5,
    "MAX_CHARS_SOLUSI": 3000,     # batas karakter teks solusi yang dikirim ke Groq
}

# Folder project utama (parent dari folder STEP ini)
PROJECT_ROOT = Path.cwd().parent
STEP1_DIR    = PROJECT_ROOT / "STEP 1"
STEP3_DIR    = PROJECT_ROOT / "STEP 3"

DATA_PROC    = STEP1_DIR / "data" / "processed"
DATA_EVAL    = STEP1_DIR / "data" / "eval"
DATA_RESULT  = Path.cwd() / "data" / "results"
MODEL_DIR    = STEP3_DIR / "models"
LOGS_DIR     = PROJECT_ROOT / "logs"

DATA_RESULT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOGS_DIR / "solution_reuse.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print("✅ Konfigurasi selesai")

## Cell 4 — Load Artefak dari Tahap 3

In [ ]:
# ── Load cases (lengkap dengan teks_full) ──────────────────
json_path = DATA_PROC / "cases.json"
assert json_path.exists(), f"❌ {json_path} tidak ditemukan. Jalankan Tahap 2 dulu."

with open(json_path, encoding="utf-8") as f:
    cases = json.load(f)

df = pd.DataFrame(cases)
print(f"✅ {len(df)} kasus dimuat dari cases.json")

# ── Load TF-IDF ─────────────────────────────────────────────
tfidf    = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
X_tfidf  = joblib.load(MODEL_DIR / "tfidf_matrix.pkl")
print(f"✅ TF-IDF dimuat — matrix: {X_tfidf.shape}")

# ── Load SentenceTransformer embeddings ─────────────────────
embeddings = np.load(MODEL_DIR / "embeddings.npy")
print(f"✅ Embeddings dimuat — shape: {embeddings.shape}")

# ── Load SentenceTransformer model ──────────────────────────
ST_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"\n🔄 Loading SentenceTransformer model...")
st_model = SentenceTransformer(ST_MODEL)
print(f"✅ ST model siap")

# ── Buat dict solusi per case_id ─────────────────────────────
# Solusi = amar putusan + vonis (ini yang akan di-reuse)
case_solutions = {}
for _, row in df.iterrows():
    cid = row["case_id"]
    case_solutions[cid] = {
        "amar_putusan"   : str(row.get("amar_putusan", "")),
        "pasal_terbukti" : str(row.get("pasal_terbukti", "")),
        "vonis_penjara"  : str(row.get("vonis_penjara", "")),
        "vonis_denda"    : str(row.get("vonis_denda", "")),
        "label"          : str(row.get("label", "")),
        "ringkasan_fakta": str(row.get("ringkasan_fakta", "")),
    }

print(f"✅ {len(case_solutions)} solusi kasus tersedia")

## Cell 5 — Koneksi Groq API

> Groq dipakai untuk merangkum solusi dari top-k kasus menjadi prediksi yang koheren.  
> Jika tidak mau pakai Groq, Cell ini bisa dilewati — fallback tersedia di Cell 7.

In [ ]:
GROQ_API_KEY = getpass("Masukkan Groq API Key (Enter untuk skip): ")

if GROQ_API_KEY.strip():
    try:
        groq_client = Groq(api_key=GROQ_API_KEY)
        test = groq_client.chat.completions.create(
            model    = CONFIG["GROQ_MODEL"],
            messages = [{"role": "user", "content": "Balas: OK"}],
            max_tokens = 5,
        )
        GROQ_AKTIF = True
        print(f"✅ Groq terhubung — mode: LLM-assisted")
    except Exception as e:
        GROQ_AKTIF = False
        print(f"⚠️  Groq gagal: {e}\n   Akan pakai fallback rule-based")
else:
    GROQ_AKTIF = False
    groq_client = None
    print("⏭️  Groq dilewati — mode: rule-based fallback")

## Cell 6 — Fungsi Retrieve (diulang dari Tahap 3)

In [ ]:
STOPWORDS_ID = set([
    "yang","dan","di","ke","dari","dengan","untuk","pada","dalam",
    "adalah","ini","itu","atau","juga","sudah","oleh","tidak","ada",
    "bahwa","telah","akan","maka","serta","para","kepada","karena",
    "atas","sebagai","dapat","tersebut","sehingga","antara","nomor",
    "halaman","republik","indonesia","mahkamah","pengadilan","agung",
])

def preprocess_teks(teks: str, max_chars: int = 8000) -> str:
    if not teks or not isinstance(teks, str): return ""
    teks = teks[:max_chars].lower()
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    kata = [k for k in teks.split() if k not in STOPWORDS_ID and len(k) > 2]
    return " ".join(kata)

def buat_teks_representasi(row) -> str:
    bagian = []
    for field in ["pasal_dakwaan","pasal_terbukti","amar_putusan"]:
        val = str(row.get(field,"")).strip()
        if val:
            bagian.append(val); bagian.append(val)
    for field in ["ringkasan_fakta","barang_bukti","terdakwa"]:
        val = str(row.get(field,"")).strip()
        if val: bagian.append(val)
    teks_penuh = str(row.get("teks_full",""))[:3000]
    if teks_penuh: bagian.append(teks_penuh)
    return " ".join(bagian)

def retrieve(query: str, k: int = None, mode: str = None) -> list:
    if k    is None: k    = CONFIG["TOP_K"]
    if mode is None: mode = CONFIG["RETRIEVAL_MODE"]

    query_clean = preprocess_teks(query)
    if not query_clean: return []

    if mode == "tfidf":
        q_vec  = tfidf.transform([query_clean])
        scores = cosine_similarity(q_vec, X_tfidf).flatten()
    elif mode == "embedding":
        q_emb  = st_model.encode([query], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, embeddings).flatten()
    else:
        raise ValueError(f"Mode tidak dikenal: {mode}")

    top_idx = np.argsort(scores)[::-1][:k]
    hasil = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[idx]
        hasil.append({
            "rank"           : rank,
            "case_id"        : row["case_id"],
            "score"          : round(float(scores[idx]), 4),
            "no_perkara"     : str(row.get("no_perkara","")),
            "pengadilan"     : str(row.get("pengadilan","")),
            "terdakwa"       : str(row.get("terdakwa","")),
            "pasal_dakwaan"  : str(row.get("pasal_dakwaan","")),
            "pasal_terbukti" : str(row.get("pasal_terbukti","")),
            "amar_putusan"   : str(row.get("amar_putusan","")),
            "vonis_penjara"  : str(row.get("vonis_penjara","")),
            "label"          : str(row.get("label","")),
            "ringkasan_fakta": str(row.get("ringkasan_fakta","")),
        })
    return hasil

print("✅ Fungsi retrieve() siap")

## Cell 7 — Algoritma Prediksi

### Dua algoritma yang diimplementasikan:
1. **Majority Vote** — pilih solusi yang paling banyak muncul di top-k
2. **Weighted Similarity** — bobot = skor cosine similarity, solusi terbaik dipilih berdasarkan total bobot

In [ ]:
def majority_vote(top_k_hasil: list) -> dict:
    """
    Pilih solusi berdasarkan label yang paling banyak muncul di top-k.
    Jika seri, ambil yang skor tertinggi.
    """
    if not top_k_hasil:
        return {"label": "tidak_diketahui", "amar_putusan": "", "vonis_penjara": ""}

    label_count = Counter(h["label"] for h in top_k_hasil)
    label_terpilih = label_count.most_common(1)[0][0]

    # Dari kasus berlabel sama, ambil yang skor tertinggi
    kandidat = [h for h in top_k_hasil if h["label"] == label_terpilih]
    best     = max(kandidat, key=lambda h: h["score"])

    solusi = case_solutions.get(best["case_id"], {})
    return {
        "metode"         : "majority_vote",
        "label_pred"     : label_terpilih,
        "confidence"     : round(label_count[label_terpilih] / len(top_k_hasil), 2),
        "best_case_id"   : best["case_id"],
        "best_score"     : best["score"],
        "amar_putusan"   : solusi.get("amar_putusan", ""),
        "pasal_terbukti" : solusi.get("pasal_terbukti", ""),
        "vonis_penjara"  : solusi.get("vonis_penjara", ""),
        "vonis_denda"    : solusi.get("vonis_denda", ""),
        "distribusi_label": dict(label_count),
    }


def weighted_similarity(top_k_hasil: list) -> dict:
    """
    Pilih solusi berdasarkan total bobot similarity per label.
    Label dengan total skor tertinggi menang.
    """
    if not top_k_hasil:
        return {"label": "tidak_diketahui", "amar_putusan": "", "vonis_penjara": ""}

    bobot_label = {}
    for h in top_k_hasil:
        label = h["label"]
        bobot_label[label] = bobot_label.get(label, 0) + h["score"]

    label_terpilih = max(bobot_label, key=bobot_label.get)

    kandidat = [h for h in top_k_hasil if h["label"] == label_terpilih]
    best     = max(kandidat, key=lambda h: h["score"])

    total_bobot = sum(bobot_label.values())
    solusi = case_solutions.get(best["case_id"], {})
    return {
        "metode"         : "weighted_similarity",
        "label_pred"     : label_terpilih,
        "confidence"     : round(bobot_label[label_terpilih] / total_bobot, 2) if total_bobot else 0,
        "best_case_id"   : best["case_id"],
        "best_score"     : best["score"],
        "amar_putusan"   : solusi.get("amar_putusan", ""),
        "pasal_terbukti" : solusi.get("pasal_terbukti", ""),
        "vonis_penjara"  : solusi.get("vonis_penjara", ""),
        "vonis_denda"    : solusi.get("vonis_denda", ""),
        "bobot_per_label": {k: round(v, 4) for k, v in bobot_label.items()},
    }


def rangkum_via_groq(top_k_hasil: list, query: str) -> str:
    """
    Minta Groq merangkum solusi dari top-k kasus menjadi
    prediksi yang koheren dan mudah dibaca.
    Hanya dipanggil jika GROQ_AKTIF = True.
    """
    if not GROQ_AKTIF or not groq_client:
        return ""

    # Susun konteks dari top-k
    konteks_list = []
    for h in top_k_hasil[:3]:  # batasi 3 kasus agar tidak terlalu panjang
        sol = case_solutions.get(h["case_id"], {})
        konteks_list.append(
            f"Kasus {h['rank']} (skor={h['score']:.3f}):\n"
            f"  Pasal terbukti : {sol.get('pasal_terbukti','-')}\n"
            f"  Amar putusan   : {sol.get('amar_putusan','-')[:200]}\n"
            f"  Vonis penjara  : {sol.get('vonis_penjara','-')}\n"
            f"  Fakta          : {sol.get('ringkasan_fakta','-')[:150]}"
        )
    konteks = "\n\n".join(konteks_list)

    prompt = f"""Kamu adalah asisten analisis hukum perlindungan anak Indonesia.

Berdasarkan {len(top_k_hasil)} kasus serupa berikut, prediksi putusan yang paling mungkin
untuk kasus baru di bawah ini. Jawab dalam 3-4 kalimat: pasal yang kemungkinan terbukti,
jenis putusan (terbukti/bebas/lepas), dan estimasi vonis jika ada.

KASUS SERUPA:
{konteks}

KASUS BARU:
{query[:500]}

Prediksi putusan:"""

    try:
        import time
        resp = groq_client.chat.completions.create(
            model      = CONFIG["GROQ_MODEL"],
            messages   = [{"role": "user", "content": prompt}],
            max_tokens = 300,
            temperature= 0.2,
        )
        time.sleep(CONFIG["GROQ_DELAY"])
        return resp.choices[0].message.content.strip()
    except Exception as e:
        logger.warning(f"Groq rangkum gagal: {e}")
        return ""


print("✅ Fungsi majority_vote, weighted_similarity, rangkum_via_groq siap")

## Cell 8 — Fungsi `predict_outcome()` Terpadu

In [ ]:
def predict_outcome(query: str, k: int = None, mode: str = None) -> dict:
    """
    Pipeline lengkap prediksi solusi untuk satu kasus baru.

    Args:
        query : teks deskripsi kasus baru
        k     : jumlah kasus mirip yang diambil
        mode  : 'tfidf' atau 'embedding'

    Returns dict berisi:
        - top_k_cases       : list kasus mirip
        - majority_vote     : prediksi via majority vote
        - weighted_sim      : prediksi via weighted similarity
        - groq_summary      : ringkasan Groq (jika aktif)
        - predicted_label   : label final (dari weighted similarity)
        - predicted_solution: teks solusi final
    """
    if k    is None: k    = CONFIG["TOP_K"]
    if mode is None: mode = CONFIG["RETRIEVAL_MODE"]

    # Step 1: Retrieve top-k kasus
    top_k = retrieve(query, k=k, mode=mode)
    if not top_k:
        return {"error": "Tidak ada kasus ditemukan"}

    # Step 2: Prediksi via dua algoritma
    pred_mv = majority_vote(top_k)
    pred_ws = weighted_similarity(top_k)

    # Step 3: Rangkum via Groq (opsional)
    groq_summary = rangkum_via_groq(top_k, query) if GROQ_AKTIF else ""

    # Step 4: Pilih prediksi final
    # Pakai weighted_similarity sebagai default karena lebih nuanced
    label_final   = pred_ws["label_pred"]
    solusi_final  = pred_ws["amar_putusan"] if pred_ws["amar_putusan"] else pred_mv["amar_putusan"]

    # Jika Groq tersedia, pakai ringkasan Groq sebagai solusi final
    if groq_summary:
        solusi_final = groq_summary

    return {
        "query"              : query[:200],
        "mode"               : mode,
        "top_k_cases"        : [h["case_id"] for h in top_k],
        "top_k_scores"       : [h["score"] for h in top_k],
        "top_k_labels"       : [h["label"] for h in top_k],
        "majority_vote"      : pred_mv,
        "weighted_sim"       : pred_ws,
        "groq_summary"       : groq_summary,
        "predicted_label"    : label_final,
        "predicted_solution" : solusi_final,
    }


print("✅ Fungsi predict_outcome() siap")

## Cell 9 — Demo Manual: 5 Kasus Baru

> Sesuaikan teks kasus di bawah dengan domain Perlindungan Anak yang kamu pakai.  
> Idealnya gunakan teks dari putusan yang belum masuk corpus (kasus di luar dataset).

In [ ]:
# ============================================================
# 5 KASUS BARU UNTUK DEMO
# Ganti dengan teks kasus nyata jika tersedia
# ============================================================
KASUS_BARU = [
    {
        "query_id"   : "demo_001",
        "query_text" : (
            "Terdakwa melakukan persetubuhan dengan anak berusia 13 tahun "
            "yang merupakan tetangganya. Kejadian berlangsung beberapa kali "
            "di rumah terdakwa. Korban melapor kepada orang tua setelah "
            "mengalami trauma. Dakwaan pasal 76D UU No. 35 Tahun 2014."
        ),
        "label_sebenarnya": "terbukti"
    },
    {
        "query_id"   : "demo_002",
        "query_text" : (
            "Terdakwa didakwa melakukan kekerasan fisik terhadap anak "
            "kandungnya berusia 8 tahun hingga mengalami luka memar di "
            "beberapa bagian tubuh. Terdakwa mengaku memukul karena anak "
            "nakal. Dakwaan pasal 76C jo pasal 80 UU Perlindungan Anak."
        ),
        "label_sebenarnya": "terbukti"
    },
    {
        "query_id"   : "demo_003",
        "query_text" : (
            "Terdakwa merekrut anak usia 15 tahun untuk bekerja di tempat "
            "hiburan malam dengan iming-iming gaji besar. Korban kemudian "
            "dieksploitasi secara ekonomi. Dakwaan pasal 76I UU No. 35 "
            "Tahun 2014 tentang eksploitasi anak."
        ),
        "label_sebenarnya": "terbukti"
    },
    {
        "query_id"   : "demo_004",
        "query_text" : (
            "Terdakwa dituduh melakukan pencabulan terhadap anak tiri "
            "berusia 11 tahun. Namun berdasarkan keterangan saksi dan "
            "visum, tidak ditemukan bukti yang cukup untuk membuktikan "
            "dakwaan secara meyakinkan."
        ),
        "label_sebenarnya": "bebas"
    },
    {
        "query_id"   : "demo_005",
        "query_text" : (
            "Terdakwa orang tua yang meninggalkan anak usia 6 tahun "
            "sendirian di rumah selama 3 hari tanpa makanan dan perawatan. "
            "Anak ditemukan tetangga dalam kondisi lemah. Dakwaan "
            "penelantaran anak pasal 76B UU Perlindungan Anak."
        ),
        "label_sebenarnya": "terbukti"
    },
]

# ── Jalankan prediksi ────────────────────────────────────────
print(f"{'='*60}")
print(f"  DEMO PREDIKSI — {len(KASUS_BARU)} KASUS BARU")
print(f"{'='*60}\n")

semua_prediksi = []

for kasus in KASUS_BARU:
    print(f"\n📋 [{kasus['query_id']}]")
    print(f"   Teks: {kasus['query_text'][:80]}...")

    hasil = predict_outcome(kasus["query_text"])

    print(f"   ─────────────────────────────────────────")
    print(f"   Top-5 cases   : {hasil['top_k_cases']}")
    print(f"   Top-5 scores  : {hasil['top_k_scores']}")
    print(f"   Top-5 labels  : {hasil['top_k_labels']}")
    print(f"   ─────────────────────────────────────────")
    print(f"   Majority vote : {hasil['majority_vote']['label_pred']} "
          f"(conf={hasil['majority_vote']['confidence']})")
    print(f"   Weighted sim  : {hasil['weighted_sim']['label_pred']} "
          f"(conf={hasil['weighted_sim']['confidence']})")
    print(f"   Label sebenarnya: {kasus['label_sebenarnya']}")
    print(f"   ─────────────────────────────────────────")
    if hasil.get("groq_summary"):
        print(f"   Groq summary  : {hasil['groq_summary'][:200]}...")
    else:
        print(f"   Prediksi final: {hasil['predicted_solution'][:150]}...")

    # Cek apakah prediksi benar
    benar = hasil["predicted_label"] == kasus["label_sebenarnya"]
    print(f"   ✅ Prediksi {'BENAR' if benar else '❌ SALAH'}")

    # Simpan untuk CSV
    semua_prediksi.append({
        "query_id"           : kasus["query_id"],
        "query_text"         : kasus["query_text"][:200],
        "label_sebenarnya"   : kasus["label_sebenarnya"],
        "predicted_label"    : hasil["predicted_label"],
        "benar"              : benar,
        "top_5_case_ids"     : ", ".join(hasil["top_k_cases"]),
        "top_5_scores"       : ", ".join(str(s) for s in hasil["top_k_scores"]),
        "majority_vote_label": hasil["majority_vote"]["label_pred"],
        "majority_confidence": hasil["majority_vote"]["confidence"],
        "weighted_sim_label" : hasil["weighted_sim"]["label_pred"],
        "weighted_confidence": hasil["weighted_sim"]["confidence"],
        "predicted_solution" : hasil["predicted_solution"][:300],
        "groq_summary"       : hasil.get("groq_summary", "")[:300],
    })

# Akurasi demo
n_benar = sum(1 for p in semua_prediksi if p["benar"])
print(f"\n{'='*60}")
print(f"  AKURASI DEMO: {n_benar}/{len(semua_prediksi)} = {n_benar/len(semua_prediksi):.0%}")
print(f"{'='*60}")

## Cell 10 — Simpan Hasil Prediksi

In [ ]:
# ── Simpan predictions.csv ─────────────────────────────────
df_pred = pd.DataFrame(semua_prediksi)
pred_csv = DATA_RESULT / "predictions.csv"
df_pred.to_csv(pred_csv, index=False, encoding="utf-8-sig")
print(f"✅ predictions.csv disimpan: {pred_csv}")
print(f"   Kolom: {list(df_pred.columns)}")

# ── Simpan predictions.json (lengkap) ──────────────────────
pred_json = DATA_RESULT / "predictions.json"
with open(pred_json, "w", encoding="utf-8") as f:
    json.dump(semua_prediksi, f, ensure_ascii=False, indent=2)
print(f"✅ predictions.json disimpan: {pred_json}")

# ── Preview tabel ───────────────────────────────────────────
print(f"\n📊 Preview predictions.csv:")
print(df_pred[["query_id","label_sebenarnya","predicted_label","benar",
               "majority_vote_label","weighted_sim_label"]].to_string(index=False))

## Cell 11 — (Opsional) Revisi & Retain

> Tambahkan kasus baru yang terbukti prediksinya benar ke dalam case base.  
> Ini mengimplementasikan siklus **Revise & Retain** dari CBR.

In [ ]:
def retain_kasus_baru(kasus_baru: dict, kasus_teks: str, force: bool = False) -> bool:
    """
    Tambahkan kasus baru ke cases.json jika prediksinya terbukti benar.

    Args:
        kasus_baru  : dict hasil predict_outcome()
        kasus_teks  : teks lengkap kasus
        force       : tambahkan meski prediksi salah (untuk review manual)

    Returns: True jika berhasil ditambahkan
    """
    if not force and not kasus_baru.get("benar", False):
        print(f"⏭️  Kasus dilewati — prediksi salah (gunakan force=True untuk override)")
        return False

    # Load cases saat ini
    with open(DATA_PROC / "cases.json", encoding="utf-8") as f:
        cases_existing = json.load(f)

    # Buat case_id baru
    new_id = f"case_new_{len(cases_existing)+1:03d}"

    new_case = {
        "case_id"         : new_id,
        "url_detail"      : "",
        "path_txt"        : "",
        "no_perkara"      : "",
        "tanggal_putusan" : "",
        "pengadilan"      : "",
        "terdakwa"        : "",
        "jaksa"           : "",
        "pasal_dakwaan"   : "",
        "pasal_terbukti"  : kasus_baru.get("weighted_sim", {}).get("pasal_terbukti", ""),
        "amar_putusan"    : kasus_baru.get("predicted_solution", ""),
        "vonis_penjara"   : kasus_baru.get("weighted_sim", {}).get("vonis_penjara", ""),
        "vonis_denda"     : "",
        "barang_bukti"    : "",
        "ringkasan_fakta" : kasus_baru.get("query", "")[:300],
        "jumlah_kata"     : len(kasus_teks.split()),
        "label"           : kasus_baru.get("predicted_label", ""),
        "sumber_ekstraksi": "retain",
        "teks_full"       : kasus_teks,
    }

    cases_existing.append(new_case)

    with open(DATA_PROC / "cases.json", "w", encoding="utf-8") as f:
        json.dump(cases_existing, f, ensure_ascii=False, indent=2)

    print(f"✅ Kasus baru {new_id} ditambahkan ke case base")
    print(f"   Total kasus sekarang: {len(cases_existing)}")
    print(f"   ⚠️  Jalankan ulang Tahap 3 untuk update model retrieval")
    return True


# Contoh: retain kasus demo_001 jika prediksinya benar
pred_001 = [p for p in semua_prediksi if p["query_id"] == "demo_001"]
if pred_001 and pred_001[0]["benar"]:
    kasus_teks_001 = KASUS_BARU[0]["query_text"]
    hasil_001 = predict_outcome(KASUS_BARU[0]["query_text"])
    retain_kasus_baru(hasil_001, kasus_teks_001)
else:
    print("⏭️  demo_001 tidak di-retain (prediksi salah atau tidak dijalankan)")

---
## ✅ Tahap 4 Selesai!

**Output yang dihasilkan:**
```
/data/results/
├── predictions.csv    ← hasil prediksi semua kasus demo
└── predictions.json   ← versi lengkap
/logs/
└── solution_reuse.log
```

**Struktur `predictions.csv`:**
| Kolom | Keterangan |
|-------|-----------|
| query_id | ID kasus uji |
| label_sebenarnya | Label ground-truth |
| predicted_label | Label hasil prediksi |
| benar | True/False |
| top_5_case_ids | 5 kasus paling mirip |
| majority_vote_label | Prediksi majority vote |
| weighted_sim_label | Prediksi weighted similarity |
| predicted_solution | Teks solusi prediksi |

**Siklus CBR yang sudah selesai:**
```
Retrieve ✅ → Reuse ✅ → Revise ✅ → Retain ✅
```

**Langkah selanjutnya:** Buka `05_evaluation.ipynb` untuk evaluasi lengkap.